In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from typing_extensions import TypedDict, Annotated
from langgraph.graph import add_messages
import json
import sqlite3
import datetime
import warnings
warnings.filterwarnings('ignore')

client = OpenAI()
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("All imports successful")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
# Cell 2 — Load Resume + Job Description + Tailored Question Bank
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

resume_text = """
Gurveer Sidhu — Data Science Student, Wentworth Institute of Technology
Skills: Python, SQL, R, C++, XGBoost, LangGraph, LangChain, PyTorch, 
FAISS, ChromaDB, Scikit-learn, SHAP, DoWhy, Power BI, Tableau, 
IBM Watsonx, OpenAI API, Ollama, Gradio, DuckDB
Projects: Portfolio Optimizer (Black-Scholes, Monte Carlo), Credit Risk 
Engine (XGBoost+SHAP), ILS Glide Path, Great Circle Routing, RAG Financial 
Analyst, Agentic SQL, Fraud Detection, Flight Delay Causal ML, ALS 
Recommender, LLM Eval Benchmark, Mechanistic Interpretability GPT-2, 
RLHF Pipeline, Multi-Agent Conversations, LangGraph Email Triage Agent
Education: B.S. Data Science Wentworth (2024-2027), A.S. Health IT 
Benjamin Franklin (2022-2024, GPA 3.5, Carnegie Award)
Experience: Registrar Admin (ERP, SEVIS, data migration), Student 
Consultant (finance, biotech, DCF modeling)
Certifications: Google Data Analytics, Google IT Professional
Other: IFR-rated pilot, Admissions Ambassador
"""

# Paste any job description here
job_description = """
PASTE YOUR JOB DESCRIPTION HERE
"""

question_bank = {
    "machine_learning": [
        {"q": "Explain the bias-variance tradeoff and how you manage it.",
         "rubric": ["defines bias/variance", "tradeoff relationship", "regularization", "cross-validation", "practical example"]},
        {"q": "Walk me through how XGBoost works internally.",
         "rubric": ["gradient boosting", "tree construction", "learning rate", "regularization terms", "compared to Random Forest"]},
        {"q": "How do you handle severely imbalanced datasets?",
         "rubric": ["SMOTE", "class weights", "evaluation metrics", "threshold tuning", "precision-recall curve"]},
        {"q": "Explain SHAP values from first principles.",
         "rubric": ["Shapley game theory", "local vs global", "TreeExplainer", "practical use case", "vs permutation importance"]},
        {"q": "When would you choose XGBoost over a neural network?",
         "rubric": ["tabular data", "interpretability", "training time", "data size", "feature engineering"]},
        {"q": "What is regularization? Compare L1 vs L2.",
         "rubric": ["L1 sparsity", "L2 weight shrinkage", "elastic net", "practical selection"]},
        {"q": "Explain cross-validation and when to use stratified k-fold.",
         "rubric": ["reduces variance", "stratified for imbalance", "time series CV", "computational cost"]},
        {"q": "What is the curse of dimensionality?",
         "rubric": ["distance metric degradation", "sparsity", "PCA", "feature selection", "regularization"]}
    ],
    "deep_learning": [
        {"q": "Explain backpropagation and the role of the chain rule.",
         "rubric": ["forward pass", "loss computation", "chain rule", "weight updates", "vanishing gradient"]},
        {"q": "Explain the transformer architecture end to end.",
         "rubric": ["tokenization", "positional encoding", "multi-head attention", "feed-forward layers", "layer norm"]},
        {"q": "What is the vanishing gradient problem and how does ResNet address it?",
         "rubric": ["gradient shrinkage", "ReLU partial solution", "skip connections", "identity mapping"]},
        {"q": "What is dropout and what problem does it solve?",
         "rubric": ["random deactivation", "prevents co-adaptation", "ensemble interpretation", "train vs inference"]},
        {"q": "Explain transfer learning and fine-tuning strategy.",
         "rubric": ["pretrained weights", "feature extraction vs fine-tuning", "data size", "domain similarity", "layer freezing"]}
    ],
    "statistics_probability": [
        {"q": "Explain the Central Limit Theorem and why it matters.",
         "rubric": ["sample mean distribution", "normality as n grows", "independence", "practical inference implications"]},
        {"q": "What is a p-value? What are common misconceptions?",
         "rubric": ["probability of data given null", "not probability null is true", "significance vs effect size", "multiple testing"]},
        {"q": "Explain Bayesian vs frequentist inference.",
         "rubric": ["prior likelihood posterior", "probability as belief", "updating with evidence", "A/B testing differences"]},
        {"q": "How do you design an A/B test end to end?",
         "rubric": ["null hypothesis", "power analysis", "sample size", "randomization", "practical vs statistical significance"]},
        {"q": "What is bootstrapping and when is it preferred?",
         "rubric": ["resampling with replacement", "non-parametric", "confidence intervals", "small samples", "non-normal data"]},
        {"q": "Explain multicollinearity and how to detect it.",
         "rubric": ["correlated features", "VIF", "coefficient instability", "PCA or removal"]}
    ],
    "sql_data_engineering": [
        {"q": "Write a query for top 3 customers by revenue per region using window functions.",
         "rubric": ["RANK or DENSE_RANK", "PARTITION BY region", "ORDER BY revenue DESC", "handles ties"]},
        {"q": "Explain CTEs vs subqueries vs temp tables.",
         "rubric": ["WITH clause", "readability", "recursive CTEs", "performance differences", "when to use each"]},
        {"q": "How do you optimize a slow query on a 100M row table?",
         "rubric": ["indexing", "avoid SELECT *", "query plan", "partitioning", "materialized views"]},
        {"q": "Explain ACID properties and why they matter.",
         "rubric": ["atomicity consistency isolation durability", "transaction integrity", "failure recovery", "pipeline implications"]},
        {"q": "OLTP vs OLAP — when do you use each?",
         "rubric": ["transactional vs analytical", "normalization vs denormalization", "row vs columnar", "examples"]},
        {"q": "How would you design a pipeline for 10 billion rows daily?",
         "rubric": ["batch vs streaming", "partitioning", "Spark", "idempotency", "monitoring"]}
    ],
    "ai_safety_alignment": [
        {"q": "What is the AI alignment problem?",
         "rubric": ["goal specification", "instrumental convergence", "scalability", "concrete examples"]},
        {"q": "Explain RLHF end to end.",
         "rubric": ["preference data", "reward model", "KL penalty", "PPO", "reward hacking limitation"]},
        {"q": "What is mechanistic interpretability?",
         "rubric": ["circuit discovery", "activation patching", "IOI task", "induction heads", "TransformerLens"]},
        {"q": "What are the main LLM failure modes in production?",
         "rubric": ["hallucination", "prompt injection", "sycophancy", "reward hacking", "distributional shift"]},
        {"q": "Explain Constitutional AI vs standard RLHF.",
         "rubric": ["self-critique loop", "principle-based feedback", "reduced human labeling", "advantages"]},
        {"q": "What is scalable oversight?",
         "rubric": ["human can't verify superhuman outputs", "debate and amplification", "weak-to-strong generalization"]}
    ],
    "llm_agentic_systems": [
        {"q": "Explain the ReAct pattern for AI agents.",
         "rubric": ["reasoning + acting loop", "tool use", "observation feedback", "vs chain-of-thought", "example"]},
        {"q": "What is RAG and when is it better than fine-tuning?",
         "rubric": ["retrieval augmented generation", "knowledge cutoff", "cost", "dynamic knowledge", "hallucination reduction"]},
        {"q": "How do you evaluate a RAG pipeline?",
         "rubric": ["retrieval precision recall", "faithfulness", "answer relevance", "RAGAS", "human eval"]},
        {"q": "Explain LangGraph StateGraph for stateful agents.",
         "rubric": ["typed state", "nodes and edges", "conditional routing", "persistence", "vs simple chains"]},
        {"q": "What are challenges in multi-agent systems?",
         "rubric": ["coordination", "shared state", "error propagation", "turn orchestration", "communication overhead"]},
        {"q": "FAISS vs ChromaDB — tradeoffs.",
         "rubric": ["embedding space", "cosine vs L2", "FAISS flat vs IVF", "ChromaDB persistence", "scalability"]}
    ],
    "mlops_production": [
        {"q": "Walk me through deploying an ML model to production.",
         "rubric": ["serialization", "API serving", "containerization", "CI/CD", "monitoring", "rollback"]},
        {"q": "What is model drift and how do you handle it?",
         "rubric": ["data drift vs concept drift", "monitoring", "statistical tests", "retraining triggers", "shadow deployment"]},
        {"q": "Online vs batch inference — when do you use each?",
         "rubric": ["latency requirements", "throughput", "examples", "cost tradeoffs"]},
        {"q": "How do you set up ML experiment tracking?",
         "rubric": ["MLflow", "parameters metrics artifacts", "reproducibility", "model registry", "collaboration"]}
    ],
    "finance_quantitative": [
        {"q": "How would you build a credit risk model from scratch?",
         "rubric": ["data collection", "feature engineering", "model selection", "ROC-AUC", "scorecard calibration"]},
        {"q": "Explain Black-Scholes-Merton from first principles.",
         "rubric": ["GBM assumption", "Itô's Lemma", "inputs", "Greeks", "limitations"]},
        {"q": "What is VaR and what are its limitations?",
         "rubric": ["confidence level", "time horizon", "tail risk", "normality assumption", "CVaR alternative"]},
        {"q": "Explain the Markowitz efficient frontier.",
         "rubric": ["expected return and covariance", "diversification", "efficient frontier shape", "estimation error limitation"]}
    ],
    "aviation_data_science": [
        {"q": "How would you build a flight delay prediction system?",
         "rubric": ["features: weather NAS carrier", "causal framing", "real-time vs batch", "evaluation", "operational integration"]},
        {"q": "Explain ILS glide slope mathematics.",
         "rubric": ["3 degree angle", "DDM", "ICAO Annex 10", "deviation in dots", "decision height"]},
        {"q": "How do you use causal inference to separate weather from carrier delays?",
         "rubric": ["causal DAG", "confounders", "backdoor criterion", "DoWhy", "ATE interpretation"]},
        {"q": "How would you optimize fuel burn for a transoceanic route?",
         "rubric": ["great circle geometry", "wind correction", "ETOPS constraints", "fuel model", "cost index"]}
    ]
}

total = sum(len(v) for v in question_bank.values())
print(f"Question bank: {total} questions across {len(question_bank)} domains")
for domain, qs in question_bank.items():
    print(f"  {domain:<25} {len(qs)} questions")
print(f"\nResume loaded: {len(resume_text.split())} words")
print(f"JD loaded: {len(job_description.split())} words")

Question bank: 49 questions across 9 domains
  machine_learning          8 questions
  deep_learning             5 questions
  statistics_probability    6 questions
  sql_data_engineering      6 questions
  ai_safety_alignment       6 questions
  llm_agentic_systems       6 questions
  mlops_production          4 questions
  finance_quantitative      4 questions
  aviation_data_science     4 questions

Resume loaded: 118 words
JD loaded: 5 words


In [3]:
# Cell 3 — JD Analyzer: Identify Skill Gaps & Select Relevant Questions
def analyze_jd_and_select_questions(resume, jd, question_bank):
    """
    Use GPT-4o-mini to analyze the JD, identify skill gaps,
    and select the most relevant interview questions.
    """
    prompt = f"""You are an expert technical recruiter and data science hiring manager.

RESUME:
{resume}

JOB DESCRIPTION:
{jd}

Your tasks:
1. Identify the top 5 technical skills the JD requires that are WEAK or MISSING from the resume
2. Identify the top 3 strengths the candidate has that match the JD
3. Select the 8 most relevant interview question domains from this list: {list(question_bank.keys())}
4. Give a fit score from 0-100 for this candidate for this role
5. List 3 specific topics the interviewer should probe deeply

Return ONLY valid JSON in this exact format:
{{
  "fit_score": 72,
  "strengths": ["strength1", "strength2", "strength3"],
  "gaps": ["gap1", "gap2", "gap3", "gap4", "gap5"],
  "relevant_domains": ["domain1", "domain2", "domain3", "domain4", "domain5", "domain6", "domain7", "domain8"],
  "probe_topics": ["topic1", "topic2", "topic3"],
  "jd_summary": "One sentence summary of what this role needs"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    
    import json
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

print("Analyzing JD vs Resume...")
analysis = analyze_jd_and_select_questions(resume_text, job_description, question_bank)

print(f"\n{'='*55}")
print(f"  JD ANALYSIS REPORT")
print(f"{'='*55}")
print(f"  Role fit score:    {analysis['fit_score']}/100")
print(f"\n  JD Summary:")
print(f"  {analysis['jd_summary']}")
print(f"\n  Your strengths for this role:")
for s in analysis['strengths']:
    print(f"    ✓ {s}")
print(f"\n  Skill gaps to address:")
for g in analysis['gaps']:
    print(f"    ✗ {g}")
print(f"\n  Topics interviewer will probe:")
for t in analysis['probe_topics']:
    print(f"    → {t}")
print(f"\n  Most relevant domains: {analysis['relevant_domains']}")

Analyzing JD vs Resume...

  JD ANALYSIS REPORT
  Role fit score:    72/100

  JD Summary:
  The role requires a data scientist with strong machine learning skills, experience in deploying models, and a solid understanding of finance and statistics.

  Your strengths for this role:
    ✓ Proficiency in Python and SQL
    ✓ Experience with machine learning frameworks like XGBoost and PyTorch
    ✓ Hands-on project experience in finance-related data science applications

  Skill gaps to address:
    ✗ Experience with MLOps or production deployment of models
    ✗ Knowledge of AI safety and alignment principles
    ✗ Experience with large-scale data engineering
    ✗ Familiarity with advanced statistical methods
    ✗ Experience in real-time data processing or streaming data

  Topics interviewer will probe:
    → Implementation of machine learning models in production
    → Understanding of causal inference methods
    → Experience with data visualization tools and their application in b

In [4]:
# Cell 4 — Build Tailored Question List
import random

def build_session_questions(analysis, question_bank, n_questions=8):
    session_questions = []
    relevant = analysis['relevant_domains']
    
    for domain in relevant:
        if domain in question_bank and len(session_questions) < n_questions:
            qs = question_bank[domain]
            n = min(2, len(qs), n_questions - len(session_questions))
            selected = random.sample(qs, n)
            for q in selected:
                session_questions.append({
                    'domain': domain,
                    'question': q['q'],
                    'rubric': q['rubric']
                })
    
    random.shuffle(session_questions)
    return session_questions[:n_questions]

session_qs = build_session_questions(analysis, question_bank, n_questions=8)

print(f"Session prepared: {len(session_qs)} tailored questions")
print(f"\nYour interview questions for this role:")
for i, q in enumerate(session_qs, 1):
    print(f"\n  Q{i} [{q['domain']}]:")
    print(f"  {q['question']}")

Session prepared: 8 tailored questions

Your interview questions for this role:

  Q1 [machine_learning]:
  Walk me through how XGBoost works internally.

  Q2 [machine_learning]:
  Explain SHAP values from first principles.

  Q3 [deep_learning]:
  What is dropout and what problem does it solve?

  Q4 [sql_data_engineering]:
  How do you optimize a slow query on a 100M row table?

  Q5 [deep_learning]:
  Explain transfer learning and fine-tuning strategy.

  Q6 [sql_data_engineering]:
  How would you design a pipeline for 10 billion rows daily?

  Q7 [statistics_probability]:
  How do you design an A/B test end to end?

  Q8 [statistics_probability]:
  What is bootstrapping and when is it preferred?


In [5]:
# Cell 5 — Save as Streamlit App
app_code = """
from dotenv import load_dotenv
import os
load_dotenv(r'C:\\Users\\Gurveer\\ds-portfolio\\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import streamlit as st
import json
import sqlite3
import datetime
import pandas as pd
from openai import OpenAI

client = OpenAI()

# ── Question bank (abbreviated for app) ──────────────────────
questions = [
    {"domain": "machine_learning", "q": "Explain the bias-variance tradeoff.", 
     "rubric": ["defines bias/variance", "tradeoff relationship", "regularization", "practical example"]},
    {"domain": "sql_data_engineering", "q": "How do you optimize a slow SQL query on 100M rows?",
     "rubric": ["indexing", "query plan", "partitioning", "avoid SELECT *"]},
    {"domain": "ai_safety_alignment", "q": "Explain RLHF end to end.",
     "rubric": ["preference data", "reward model", "KL penalty", "PPO", "limitations"]},
    {"domain": "llm_agentic_systems", "q": "What is RAG and when is it better than fine-tuning?",
     "rubric": ["retrieval augmented generation", "knowledge cutoff", "cost", "dynamic knowledge"]},
    {"domain": "statistics_probability", "q": "How do you design an A/B test end to end?",
     "rubric": ["null hypothesis", "power analysis", "randomization", "practical vs statistical significance"]},
    {"domain": "mlops_production", "q": "Walk me through deploying an ML model to production.",
     "rubric": ["serialization", "API serving", "containerization", "CI/CD", "monitoring"]},
    {"domain": "finance_quantitative", "q": "Explain Black-Scholes-Merton from first principles.",
     "rubric": ["GBM assumption", "Ito Lemma", "inputs", "Greeks", "limitations"]},
    {"domain": "deep_learning", "q": "Explain the transformer architecture end to end.",
     "rubric": ["tokenization", "positional encoding", "multi-head attention", "feed-forward", "layer norm"]},
]

DB_PATH = r'C:\\Users\\Gurveer\\ds-portfolio\\project-22-interview-prep-agent\\data\\sessions.db'
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''CREATE TABLE IF NOT EXISTS sessions
        (id INTEGER PRIMARY KEY AUTOINCREMENT,
         date TEXT, domain TEXT, question TEXT,
         score INTEGER, max_score INTEGER,
         percentage REAL, feedback TEXT, missing TEXT)''')
    conn.commit()
    conn.close()

def score_answer(question, rubric, answer, domain):
    prompt = f\"\"\"You are a strict technical interviewer scoring a data science candidate.
Question: {question}
Domain: {domain}
Rubric: {json.dumps(rubric)}
Answer: {answer}
Return ONLY valid JSON:
{{
  "total": 3,
  "max": {len(rubric)},
  "percentage": 75,
  "feedback": "feedback here",
  "what_was_missing": "missing points",
  "model_answer_hint": "key points of strong answer"
}}\"\"\"
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    raw = resp.choices[0].message.content.strip().replace("```json","").replace("```","")
    return json.loads(raw)

def save_result(domain, question, score, max_score, pct, feedback, missing):
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''INSERT INTO sessions 
        (date, domain, question, score, max_score, percentage, feedback, missing)
        VALUES (?,?,?,?,?,?,?,?)''',
        (datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
         domain, question, score, max_score, pct, feedback, missing))
    conn.commit()
    conn.close()

# ── Streamlit UI ──────────────────────────────────────────────
init_db()
st.set_page_config(page_title="Interview Prep Agent", page_icon="🎯", layout="wide")
st.title("🎯 DS & AI Interview Prep Agent")
st.caption("Answer each question and get instant AI-powered feedback with rubric scoring.")

if "q_idx" not in st.session_state:
    st.session_state.q_idx = 0
if "results" not in st.session_state:
    st.session_state.results = []
if "scored" not in st.session_state:
    st.session_state.scored = False

q_idx = st.session_state.q_idx

# Progress bar
st.progress(q_idx / len(questions))
st.caption(f"Question {min(q_idx+1, len(questions))} of {len(questions)}")

if q_idx < len(questions):
    q = questions[q_idx]
    st.markdown(f"### [{q['domain'].replace('_',' ').upper()}]")
    st.markdown(f"**{q['q']}**")
    
    answer = st.text_area("Your answer:", height=180, key=f"ans_{q_idx}")
    
    col1, col2 = st.columns([1, 4])
    with col1:
        submit = st.button("Submit Answer", type="primary")
    with col2:
        skip = st.button("Skip →")
    
    if submit and answer.strip():
        with st.spinner("Scoring your answer..."):
            result = score_answer(q['q'], q['rubric'], answer, q['domain'])
        
        pct = result['percentage']
        color = "green" if pct >= 75 else "orange" if pct >= 50 else "red"
        
        st.markdown(f"### Score: :{color}[{result['total']}/{result['max']} ({pct}%)]")
        st.markdown(f"**📝 Feedback:** {result['feedback']}")
        st.markdown(f"**❌ Missing:** {result['what_was_missing']}")
        st.markdown(f"**💡 Strong answer includes:** {result['model_answer_hint']}")
        
        save_result(q['domain'], q['q'], result['total'], result['max'],
                    pct, result['feedback'], result['what_was_missing'])
        
        st.session_state.results.append({
            'domain': q['domain'], 'score': result['total'],
            'max': result['max'], 'percentage': pct
        })
        
        if st.button("Next Question →"):
            st.session_state.q_idx += 1
            st.rerun()
    
    if skip:
        st.session_state.q_idx += 1
        st.rerun()

else:
    # Session complete
    st.success("🎉 Session Complete!")
    results = st.session_state.results
    if results:
        df = pd.DataFrame(results)
        total = df['score'].sum()
        possible = df['max'].sum()
        overall = (total / possible * 100) if possible > 0 else 0
        grade = "Excellent 🏆" if overall >= 85 else "Good 👍" if overall >= 70 else "Keep Practicing 💪"
        
        st.metric("Overall Score", f"{overall:.1f}%", grade)
        
        st.markdown("### Performance by Domain")
        domain_df = df.groupby('domain')['percentage'].mean().reset_index()
        st.bar_chart(domain_df.set_index('domain'))
        
        st.markdown("### Historical Progress")
        conn = sqlite3.connect(DB_PATH)
        hist = pd.read_sql(
            "SELECT date, AVG(percentage) as avg_score FROM sessions GROUP BY date",
            conn
        )
        conn.close()
        if len(hist) > 1:
            st.line_chart(hist.set_index('date'))
        
        if st.button("Start New Session"):
            st.session_state.q_idx = 0
            st.session_state.results = []
            st.rerun()
"""

src_dir = r'C:\\Users\\Gurveer\\ds-portfolio\\project-22-interview-prep-agent\\src'
os.makedirs(src_dir, exist_ok=True)
with open(f'{src_dir}\\app.py', 'w') as f:
    f.write(app_code)
print("Streamlit app saved to src/app.py")
print("\\nTo run the interview agent:")
print("  cd C:\\\\Users\\\\Gurveer\\\\ds-portfolio\\\\project-22-interview-prep-agent\\\\src")
print("  streamlit run app.py")

Streamlit app saved to src/app.py
\nTo run the interview agent:
  cd C:\\Users\\Gurveer\\ds-portfolio\\project-22-interview-prep-agent\\src
  streamlit run app.py
